# Week 5, Notebook 3: Your First CNN - Caribbean Flags

**Training a Convolutional Neural Network to recognise six Caribbean flags.**

*Caribbean AI - Adrian Dunkley*

---

### What you will build
- A real **Convolutional Neural Network (CNN)** in Keras.
- A model that looks at a 32x32 flag and names the country: **Jamaica, Haiti,
  Barbados, Bahamas, Trinidad and Tobago, or Guyana**.
- The full computer-vision workflow: load images, build conv layers, train, read
  the curves, and test.

### The big idea
You now know images are grids of numbers (Notebook 1) and that filters find
patterns (Notebook 2). A **CNN** ties these together: it **learns its own
filters** to find whatever patterns best tell the flags apart, then a few normal
neural-network layers make the final decision. Keras builds the whole thing from
a short description.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.load("data/caribbean_flags.npz", allow_pickle=True)
X = data["X"].astype("float32") / 255.0    # scale pixels to 0..1
y = data["y"]
class_names = list(data["class_names"])
print("Images:", X.shape, " Labels:", y.shape)
print("Flags:", class_names)

### Step 1: Look at the data first

Always eyeball your images before training. Here is one example of each flag.
They are small (32x32) and a little noisy on purpose, just like real photos are
never perfectly clean.

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
for label, ax in enumerate(axes):
    idx = np.where(y == label)[0][0]
    ax.imshow(X[idx]); ax.set_title(class_names[label], fontsize=9); ax.axis("off")
plt.suptitle("One example of each Caribbean flag")
plt.show()

### Step 2: Split into training and test sets

Same habit as always: train on most of the images, keep some hidden to test on.
`stratify=y` makes sure every flag is represented fairly in both piles.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Training images:", len(X_train), " Test images:", len(X_test))

baseline = np.bincount(y).max() / len(y)
print(f"Baseline (always guess the most common flag): {baseline:.1%}")

### Step 3: Build the CNN

Read the layers top to bottom, like a pipeline the image flows through:

- **Conv2D(16, 3)**: 16 filters, each 3x3, that the network will *learn* (just
  like the hand-made edge filter from Notebook 2, but 16 of them and learned).
- **MaxPooling2D**: shrink the image by half, keeping the strongest signals. This
  makes the model faster and less fussy about exact position.
- A second **Conv2D + MaxPooling** pair to find bigger patterns.
- **Flatten**: unroll the grid into a long list of numbers.
- **Dense** layers: ordinary neurons (Week 4) that make the final choice.
- **softmax**: turn the 6 outputs into probabilities that add to 100%.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(16, 3, activation="relu"),   # learn 16 small filters
    layers.MaxPooling2D(),                       # shrink by half
    layers.Conv2D(32, 3, activation="relu"),   # learn 32 bigger-pattern filters
    layers.MaxPooling2D(),
    layers.Flatten(),                            # grid -> list
    layers.Dense(64, activation="relu"),
    layers.Dense(6, activation="softmax"),       # 6 flags -> 6 probabilities
])
model.summary()

### Step 4: Compile and train

Same recipe as Week 4: the `adam` optimiser, a `sparse_categorical_crossentropy`
loss for multi-class picking, and `accuracy` to watch. Each **epoch** is one pass
over the training flags.

In [ ]:
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

history = model.fit(X_train, y_train,
                    validation_split=0.15,
                    epochs=10, batch_size=32, verbose=0)
print("Final training accuracy:  ", round(history.history["accuracy"][-1], 3))
print("Final validation accuracy:", round(history.history["val_accuracy"][-1], 3))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history["loss"], label="training")
ax1.plot(history.history["val_loss"], label="validation")
ax1.set_title("Loss (lower is better)"); ax1.set_xlabel("epoch"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history.history["accuracy"], label="training")
ax2.plot(history.history["val_accuracy"], label="validation")
ax2.set_title("Accuracy (higher is better)"); ax2.set_xlabel("epoch"); ax2.legend(); ax2.grid(alpha=0.3)
plt.show()

### Step 5: Test on unseen flags

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.1%}")
print(f"Baseline:      {baseline:.1%}")
print("The CNN can read Caribbean flags!" if test_acc > 0.9 else "Needs more training.")

### Step 6: See its predictions

Let us show a handful of test flags with the CNN's guess and how confident it
was. Green titles are correct, red are mistakes.

In [ ]:
probs = model.predict(X_test, verbose=0)
preds = probs.argmax(axis=1)

fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, i in zip(axes.flat, range(12)):
    ax.imshow(X_test[i])
    guess = class_names[preds[i]]
    truth = class_names[y_test[i]]
    conf = probs[i].max()
    ok = guess == truth
    ax.set_title(f"{guess}\n({conf:.0%})", color="green" if ok else "red", fontsize=8)
    ax.axis("off")
plt.suptitle("CNN predictions on unseen flags (green = correct)")
plt.show()

### Step 7: The confusion matrix

Which flags, if any, does the CNN mix up? A confusion matrix shows the true flag
(rows) against the predicted flag (columns). A perfect model has all its counts
on the diagonal.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, preds)
fig, ax = plt.subplots(figsize=(6.5, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(6)); ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(6)); ax.set_yticklabels(class_names, fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title("Confusion matrix")
for i in range(6):
    for j in range(6):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout()
plt.show()

### What you learned

- A **CNN** = **learned convolution filters** (find patterns) + **pooling**
  (shrink) + **dense layers** (decide). Keras builds it from a short recipe.
- The workflow mirrors Week 4: **split, build, compile, fit, read the curves,
  test**, only now the inputs are images.
- **MaxPooling** makes the model smaller and less fussy about exact position;
  **softmax** gives per-flag probabilities; the **confusion matrix** shows any
  mix-ups.

Next notebook we take on a harder, messier job, spotting **hurricanes** in
satellite tiles, and meet **data augmentation**, the trick that makes a vision
model tough enough for the real world.

### Try it yourself
1. Remove the second Conv2D + MaxPooling pair. Does accuracy drop?
2. Change the first layer to `Conv2D(8, 3)` then `Conv2D(64, 3)`. More filters,
   better or just slower?
3. Train for only 2 epochs, then 20. Watch the accuracy and the loss curve.